# Distributional Semantics: PPMI and Truncated SVD

"You shall know a word by the company it keeps" (Firth 1957). This notebook makes the slogan a matrix. Each word
becomes the row of a word × context co-occurrence matrix, pointwise mutual information reweights the row against
the corpus background, and a truncated SVD compresses it. We prove what each step does, and test what it buys on
a corpus whose ground truth we know.

Every number is computed by `distributional_semantics_ppmi_svd.py`, which owns them. The text is Topic 3's
training split, `generate_corpus(300, 0)`: 6,130 sentences and 63,182 word tokens over 3,425 word types. The shared
generator draws every content noun from one Pitman–Yor restaurant per sector (or a shared one), so we can rebuild
it and read off each noun's **planted sector**. That is the benchmark. Hyperparameters are chosen on half of the
nouns (dev), and every score we report is on the other half (test).

In [ ]:
import sys
from pathlib import Path

import numpy as np

here = Path.cwd()
sys.path.insert(0, str(here if (here / "distributional_semantics_ppmi_svd.py").exists()
                       else here / "notebooks" / "distributional-semantics-ppmi-svd"))
import distributional_semantics_ppmi_svd as ds
from collections import Counter

ds.test_planted_labels()
dev, test = ds.benchmark_split()
lab = ds.planted_labels()
print(f"{len(ds.sentences())} sentences, {int(ds.unigram_counts().sum())} word tokens, |V| = {len(ds.vocab())}")
print(f"{len(ds.benchmark_words(1))} sector nouns; {len(ds.benchmark_words())} seen ≥ {ds.MIN_COUNT} times:",
      dict(Counter(lab[w] for w in ds.benchmark_words())))
print(f"dev {len(dev)} nouns, test {len(test)} nouns; rare band (2–4 occurrences): {len(ds.benchmark_words(2, 4))} nouns")

## 1. Words as points

**Definition 1 (co-occurrence matrix).** For a context type — a symmetric window $L \in \{1, 2, 4, 8\}$ inside a
sentence, the whole sentence, or the whole document — $C[w, c]$ counts the ordered token pairs $(i, j)$, $i \ne j$,
in one unit with word $w$ at $i$, word $c$ at $j$, and $|i - j| \le L$ for a window. $C$ is symmetric. Write
$N = \sum C$ and $n_w = \sum_c C[w, c]$.

**Definition 2 (PMI, PPMI, SPPMI).** With $P(w, c) = C[w,c]/N$, $P(w) = n_w/N$, and the smoothed context
distribution $P_\alpha(c) = n_c^\alpha / \sum_{c'} n_{c'}^\alpha$,
$$\mathrm{PMI}_\alpha(w, c) = \log \frac{P(w, c)}{P(w)\,P_\alpha(c)},$$
which is $-\infty$ where $C[w,c] = 0$. $\mathrm{PPMI}_\alpha = \max(\mathrm{PMI}_\alpha, 0)$, and the shifted
$\mathrm{SPPMI}_{\alpha,k} = \max(\mathrm{PMI}_\alpha - \log k, 0)$ (Church and Hanks 1990; Levy and Goldberg 2014;
Levy, Goldberg, and Dagan 2015). $\alpha = 1$, $k = 1$ is plain PPMI.

**Proposition 1 (log-ratio coordinates).** (i) $P(c \mid w) = P(c)\,e^{\mathrm{PMI}(w,c)}$: a PMI row and the
marginal determine the context distribution. (ii) $\sum_c P(c \mid w)\,\mathrm{PMI}(w, c) = \mathrm{KL}(P(\cdot\mid w)\,\|\,P_C) \ge 0$,
with equality iff $w$ is independent of its contexts. (iii) $\sum_w P(w)\,\mathrm{KL}(P(\cdot \mid w)\,\|\,P_C) = I(W; C)$.

*Proof.* (i) is the definition. (ii) Substitute (i); Gibbs' inequality. (iii) is the definition of mutual information
as the divergence of the joint from the product of marginals, the identity formalRAG's `pmi-retrieval-value` proves
(its Thm 1) for answers and documents. ∎

**Theorem 1 (PMI removes the background).** Let $P$ be a context distribution with full support, and let two words
have $P(\cdot \mid w) = (1-\varepsilon_w)P + \varepsilon_w q_w$, with $q_w \ne P$. Write $f_w = q_w/P - \mathbb{1}$. As
$\varepsilon_w, \varepsilon_{w'} \to 0$: (a) the cosine of their count rows tends to 1, with $1 - \cos = O(\varepsilon^2)$;
(b) $\mathrm{PMI}_w / \varepsilon_w \to f_w$, so $\cos(\mathrm{PMI}_w, \mathrm{PMI}_{w'}) \to \cos(f_w, f_{w'})$ with error
$O(\varepsilon)$; (c) $\sum_c P(c)\,f_w(c) = 0$.

*Proof.* (a) A count row is $n_w P(\cdot \mid w) = n_w(P + \varepsilon_w(q_w - P))$, so each row's angle to $P$ is
$O(\varepsilon)$, the angle between the rows is $O(\varepsilon)$, and $1 - \cos\theta \le \theta^2/2$. (b)
$\mathrm{PMI}_w(c) = \log(1 + \varepsilon_w f_w(c)) = \varepsilon_w f_w(c) + O(\varepsilon_w^2\|f_w\|_\infty^2)$, uniformly once
$\varepsilon_w \|f_w\|_\infty \le \tfrac12$; cosine is scale-free and continuous away from 0. (c)
$\sum_c P(c)(q_w(c)/P(c) - 1) = 1 - 1$. ∎

Near the background every count row points the same way, so counts measure how corpus-like a word is. PMI
subtracts the background in log coordinates, and only the word-specific part $f_w$ is left.

In [ ]:
ds.test_cooccurrence_definition()
ds.test_pmi_log_ratio_coordinates()
ds.test_pmi_removes_background()
b = ds.background_summary()
print(f"L2, test nouns. Raw counts: mean cosine {b['counts_same_cos_mean']:.3f} within a sector, "
      f"{b['counts_cross_cos_mean']:.3f} across (median cosine of a count row with the marginal: "
      f"{b['count_marginal_cos_median']:.3f})")
print(f"PPMI: {b['ppmi_same_cos_mean']:.3f} within, {b['ppmi_cross_cos_mean']:.3f} across")

## 2. What counts cannot tell

**Proposition 2 (the ceiling, and the singleton artifact).** (i) $\mathrm{PMI}(w, c) \le \min(-\log P(w), -\log P(c))$,
with equality in the first iff $c$ occurs only with $w$; and $\mathrm{PMI}_\alpha(w,c) \le -\log P_\alpha(c)$. (ii) On a pair
seen exactly once, the plug-in $\mathrm{PMI}_\alpha(w, c) = -\log n_w - \log P_\alpha(c)$; for $\alpha = 1$ it is
$\log N - \log n_w - \log n_c$, a function of the two frequencies alone.

*Proof.* (i) $P(w, c) \le P(c)$ and $P(w, c) \le P(w)$. (ii) Substitute $C[w,c] = 1$. ∎

Topic 1's frequency spectrum, applied to pair counts rather than word counts, says how much of the matrix this is.

In [ ]:
ds.test_pmi_ceiling_and_singletons()
s = ds.singleton_summary("L2")
print(f"L2: {int(s['singletons'])} of {int(s['pairs'])} nonzero pairs are seen once ({s['singleton_share']:.1%}); "
      f"{s['ppmi_singleton_share']:.1%} of the PPMI support")
print("largest PPMI on rows of words seen …", {k[9:]: round(v, 2) for k, v in s.items() if k.startswith("max_ppmi_")})

**Proposition 3 (PPMI under independence).** Let $X \sim \mathrm{Poisson}(\mu)$ be a pair's count when $W \perp C$
with known margins, $\mu = n_w n_c / N$, so its true PMI is 0, and let $\mathrm{PPMI} = \max(\log(X/\mu), 0)$. Then
(i) $\Pr(\mathrm{PPMI} > 0) = \Pr(X > \mu)$; (ii) if $\mu < 1$, $\Pr(\mathrm{PPMI} > 0) = 1 - e^{-\mu}$, every positive value is at
least $\log(1/\mu)$, and $\mathbb{E}[\mathrm{PPMI}] \ge (1 - e^{-\mu})\log(1/\mu)$.

*Proof.* (i) $\log(X/\mu) > 0$ iff $X > \mu$. (ii) For $\mu < 1$, $X > \mu$ iff $X \ge 1$, and then
$\log(X/\mu) \ge \log(1/\mu)$. ∎

*Remark (asserted, not proved here).* As $\mu \to \infty$, $\Pr(\mathrm{PPMI} > 0) \to \tfrac12$ and
$\sqrt{2\pi\mu}\;\mathbb{E}[\mathrm{PPMI}] \to 1$, by the central limit theorem for $(X - \mu)/\sqrt\mu$. Rare pairs give
rare but large false positives, and frequent pairs frequent but tiny ones. The cure for the first kind is a
significance test (Dunning 1993), the subject of the collocations topic.

In [ ]:
ds.test_ppmi_under_independence()
n = ds.null_summary("L2")
print(f"real L2 PPMI: {int(n['real_ppmi_nnz'])} positive entries, ‖·‖²_F = {n['real_ppmi_fro2']:,.0f}; "
      f"{n['real_ppmi_mu_below_1_share']:.1%} of them have μ < 1")
print(f"shuffled tokens (every true PMI 0): {int(n['null_ppmi_nnz'])} positive entries "
      f"(Poisson independence predicts {n['poisson_expected_positives']:,.0f}), ‖·‖²_F = {n['null_ppmi_fro2']:,.0f}")
print(f"SPPMI_5: real {n['real_sppmi5_fro2']:,.0f} against shuffled {n['null_sppmi5_fro2']:,.0f}")

**Proposition 4 (context-distribution smoothing).** For $0 < \alpha < 1$,
$P_\alpha(c)/P(c) = (N/Z_\alpha)\,n_c^{\alpha - 1}$, with $Z_\alpha = \sum_c n_c^\alpha$, is strictly decreasing in $n_c$ and
equals 1 at $t^* = (N/Z_\alpha)^{1/(1-\alpha)}$. So $\mathrm{PMI}_\alpha < \mathrm{PMI}$ exactly on contexts rarer than $t^*$.
And for $k \ge k'$, $\operatorname{supp}\mathrm{SPPMI}_{\alpha,k} \subseteq \operatorname{supp}\mathrm{SPPMI}_{\alpha,k'}$.

*Proof.* $n \mapsto n^{\alpha-1}$ is strictly decreasing, and $\mathrm{PMI}_\alpha - \mathrm{PMI} = -\log(P_\alpha(c)/P(c))$. The
shift subtracts $\log k$ from every entry. ∎

Smoothing tempers rare contexts; it leaves Prop 2's singletons a function of frequency.

In [ ]:
ds.test_context_smoothing_threshold()
m = ds.smoothing_summary("L2")
print(f"L2, α = {ds.ALPHA}: t* = {m['t_star']:.1f}; {m['types_below']:.1%} of context types "
      f"({m['tokens_below']:.1%} of context tokens) are below it")
print("support sizes:", {k[4:]: int(v) for k, v in m.items() if k.startswith("nnz_")})

## 3. Truncated SVD

**Definition 3.** For $M = U\Sigma V^\top$, a rank $d$, and a weight $p \in [0, 1]$: word vectors $W_p = U_d\Sigma_d^p$ and
context vectors $C_p = V_d\Sigma_d^{1-p}$.

**Theorem 2 (what the eigenvalue weight measures).** (i) $W_pC_p^\top = M_d$ for every $p$, the best rank-$d$
approximation of $M$ (Eckart–Young–Mirsky; formalML `svd`, Thm 2). (ii) $W_pW_p^\top = U_d\Sigma_d^{2p}U_d^\top = [(MM^\top)^p]_d$,
the best rank-$d$ approximation of $(MM^\top)^p$: second-order co-occurrence for $p = 1$, the projector $U_dU_d^\top$ for
$p = 0$. (iii) If $M$ is symmetric with eigenvalues ordered by modulus, then $W_{1/2}W_{1/2}^\top = |M|_d = Q_d|\Lambda_d|Q_d^\top$,
which equals $M_d$ iff $\lambda_1, \dots, \lambda_d \ge 0$; and $\||M|_d - M_d\|_F = 2\,(\sum_{i \le d,\,\lambda_i < 0}\lambda_i^2)^{1/2}$.

*Proof.* (i) Substitute. (ii) $MM^\top = U\Sigma^2U^\top$, so $(MM^\top)^p = U\Sigma^{2p}U^\top$ is a spectral decomposition
of a PSD matrix, whose best rank-$d$ approximation keeps its top $d$ eigenpairs. (iii) By the spectral theorem
(formalML `spectral-theorem`, Thm 1) $M = Q\Lambda Q^\top$, and $\sigma_i = |\lambda_i|$, $u_i = q_i$,
$v_i = \operatorname{sign}(\lambda_i)\,q_i$ is an SVD. So $|M|_d - M_d = \sum_{i \le d,\,\lambda_i<0} 2|\lambda_i|\,q_iq_i^\top$. ∎

**Corollary 1 (adding context vectors).** For symmetric $M$, $W_{1/2} + C_{1/2} = 2\,Q_d[\Lambda_d]_+^{1/2}$, so
$\tfrac14(W_{1/2} + C_{1/2})(W_{1/2} + C_{1/2})^\top = [M_d]_+$, the positive part.

*Proof.* With $u_i = s_iq_i$ and $v_i = s_i\operatorname{sign}(\lambda_i)q_i$ for any signs $s_i$,
$\sqrt{\sigma_i}(u_i + v_i) = s_i\sqrt{|\lambda_i|}(1 + \operatorname{sign}\lambda_i)\,q_i$. ∎

In [ ]:
ds.test_svds_matches_dense_svd()
ds.test_eigenvalue_weighting()
sp = ds.spectrum_summary()
print(f"L2 PPMI is symmetric (max |M − Mᵀ| = {sp['symmetric_err']:.1e}); of its 100 eigenvalues of largest modulus "
      f"{sp['n_negative_top100']} are negative, λ₁ = {sp['lambda1']:.1f}, λ₂ = {sp['first_negative']:.1f}")
print(f"d = 100: ‖|M|_d − M_d‖_F / ‖M_d‖_F = {sp['gram_gap_rel']:.3f}")
print(f"λ₂'s eigenvector splits {sp['pos_side']} from {sp['neg_side']}; {sp['cut_mass']:.1%} of PPMI mass crosses the cut")

In [ ]:
ds.test_word_plus_context()
wc = ds.word_plus_context_summary()
print(f"Cor 1 at L2 PPMI, d = 100, p = ½ (not tuned), test nouns: W gives AUC {wc['w_auc']:.3f}, 1-NN {wc['w_nn']:.3f}; "
      f"(W + C)/2 gives AUC {wc['wc_auc']:.3f}, 1-NN {wc['wc_nn']:.3f}")

## 4. Two kinds of low rank

**Theorem 3.** Let $R = [P(w, c)/(P(w)P(c))] = e^{\circ\mathrm{PMI}}$.
(i) If $W \perp C \mid Z$ for a latent $Z \in \{1, \dots, K\}$, so that $P = \sum_z \pi_z\varphi_z\psi_z^\top$, then
$R = A\Pi B^\top$ with $A = D_w^{-1}\Phi$, $B = D_c^{-1}\Psi$ nonnegative and $\Pi = \mathrm{diag}\,\pi$; so $\operatorname{rank}R \le K$
and $R - \mathbb{1}\mathbb{1}^\top = A(\Pi - \pi\pi^\top)B^\top$ has rank at most $K - 1$.
(ii) When $\delta = \max|R - 1| < 1$, $\mathrm{PMI} = \sum_{n \ge 1}\frac{(-1)^{n+1}}{n}(R - \mathbb{1}\mathbb{1}^\top)^{\circ n}$, the $n$-th
term has rank at most $\binom{K + n - 2}{n}$, and $\|\mathrm{PMI} - (R - \mathbb{1}\mathbb{1}^\top)\|_{\max} \le \delta^2/(2(1 - \delta))$.
(iii) If $\mathrm{PMI} = XY^\top + a\mathbb{1}^\top + \mathbb{1}b^\top$ with $X, Y$ of width $d$ (a log-bilinear model), then
$\operatorname{rank}\mathrm{PMI} \le d + 2$, while $R = e^{\circ\mathrm{PMI}}$ need not be low rank.

*Proof.* (i) $P(w) = (\Phi\pi)_w$, so $A\pi = \mathbb{1}$ and likewise $B\pi = \mathbb{1}$; hence
$\mathbb{1}\mathbb{1}^\top = A\pi\pi^\top B^\top$, and $\Pi - \pi\pi^\top$ annihilates $\mathbb{1}$. (ii) The series of $\log(1 + x)$,
entrywise. A rank-$r$ matrix $\sum_{i\le r}a_ib_i^\top$ has $n$-th Hadamard power
$\sum_{i_1 \le \dots \le i_n} c_{i_1\dots i_n}(a_{i_1}\circ\cdots\circ a_{i_n})(b_{i_1}\circ\cdots\circ b_{i_n})^\top$, a sum of
$\binom{r+n-1}{n}$ rank-one terms; take $r = K - 1$. And $|\log(1+x) - x| \le \sum_{n\ge2}|x|^n/2 = x^2/(2(1-|x|))$.
(iii) Rank is subadditive. ∎

Topic models (the LDA topic) assume (i); word embeddings (the word2vec and GloVe topics) assume (iii). The two
agree to first order in the strength of association and part ways as it grows. Topic 5's Thm 6 is (iii) for
log-probabilities, and its mixture of softmaxes escapes (iii) the way (i)'s mixtures do.

In [ ]:
ds.test_mixture_ratio_low_rank()
lr, ln = ds.lowrank_summary(), ds.lowrank_summary(True)
print(f"document context, {lr['nouns']} content nouns seen ≥ {ds.LOWRANK_MIN_COUNT} times ({lr['classes']} restaurants; "
      f"{lr['zero_share']:.1%} of pairs unseen)")
print("normalized σ of R − 𝟙𝟙ᵀ:", np.round(lr["ratio_sv"][:8], 3))
print("normalized σ of PPMI:   ", np.round(lr["ppmi_sv"][:8], 3))
print(f"shuffled corpus's largest σ of R − 𝟙𝟙ᵀ, on the real scale: {ln['ratio_sv_raw'][0] / lr['ratio_sv_raw'][0]:.3f}")
print(f"energy beyond rank {ds.K_SECTORS - 1}: R − 𝟙𝟙ᵀ {lr['ratio_resid']:.1%}, PPMI {lr['ppmi_resid']:.1%}")

## 5. Recovering the planted sectors

**Definition 4 (benchmark).** The nouns the generator drew from exactly one sector restaurant and that training
shows at least five times. For a configuration (context, weighting, and for SVD $d$ and $p$), **1-NN accuracy** is the
share of nouns whose nearest other noun by cosine has the same sector, and **pair AUC** is the probability that a
random same-sector pair has a higher cosine than a random cross-sector pair (Mann–Whitney).

**Protocol.** Within each family (sparse matrices; SVD embeddings), the configuration with the highest dev AUC
over the local contexts L1–L8 and sentence, with dev 1-NN breaking ties, is selected before any test noun is
scored. The document context is excluded from selection, because the generator plants each noun's sector at the
document level, so document co-occurrence reads the label itself. It is reported as an oracle row.

In [ ]:
ds.test_auc_matches_mannwhitney()
print(f"{'context':9} {'counts AUC / 1-NN':>18} {'PPMI AUC / 1-NN':>18}   (test nouns)")
for ctx in ds.CONTEXTS:
    a, p = ds.scores((ctx, "counts", None, None)), ds.scores((ctx, "ppmi", None, None))
    tag = "  (oracle)" if ctx not in ds.LOCAL_CONTEXTS else ""
    print(f"{ctx:9} {a['test_auc']:9.3f} / {a['test_nn']:.3f} {p['test_auc']:9.3f} / {p['test_nn']:.3f}{tag}")
sel = ds.selected()
for fam, cfg in sel.items():
    print(fam, "selected on dev:", cfg, {k: round(v, 4) for k, v in ds.scores(cfg).items()})
ctx, base, d0, p0 = sel["svd"]
for p in ds.SVD_POWERS:
    print(f"  {ctx} {base}, p = {p}: test AUC by d {ds.SVD_DIMS}:",
          [round(ds.scores((ctx, base, d, p))["test_auc"], 3) for d in ds.SVD_DIMS])

In [ ]:
ds.test_recovery_on_test_words()
c = ds.comparison()
print(f"test nouns: SVD − sparse AUC = {c['auc_svd'] - c['auc_sparse']:+.4f} (jackknife se {c['auc_diff_se']:.4f}); "
      f"1-NN {c['nn_svd']:.3f} vs {c['nn_sparse']:.3f} (chance {c['test_chance_nn']:.3f})")
print(f"rare band ({c['rare_n']} nouns, never used for selection): AUC counts {c['rare_auc_counts']:.3f}, "
      f"sparse {c['rare_auc_sparse']:.3f}, SVD {c['rare_auc_svd']:.3f}; 1-NN {c['rare_nn_counts']:.3f}, "
      f"{c['rare_nn_sparse']:.3f}, {c['rare_nn_svd']:.3f} (chance {c['rare_chance_nn']:.3f})")

## 6. Finance case study: what distribution cannot see

The stand-in lexicon's positive and negative words (never the real Loughran–McDonald dictionary) fill the same
template slots: *revenue rose 6 percent, which management described as* **strong** or **weak**. In a window, an
antonym is the best substitute. Turney's semantic orientation (2002) scores a word by
$\mathrm{SO}(w) = \mathrm{PMI}(w, \mathrm{POS}) - \mathrm{PMI}(w, \mathrm{NEG})$ against two seed sets. Here it is leave-one-out, with each
seed set merged into one context and ½ added to both counts; a word with no seed co-occurrence scores 0, a coin
flip.

In [ ]:
ds.test_finance_findings()
f = ds.finance_summary()
print("L2 neighbors of 'strong':", [(w, round(s, 3)) for w, s in f["strong_neighbors_L2"]])
for ctx in ds.CONTEXTS:
    print(f"{ctx:9} polarity adjectives: cosine within {f[f'adj_within_{ctx}']:.3f}, across {f[f'adj_cross_{ctx}']:.3f} | "
          f"SO coverage {f[f'so_coverage_{ctx}']:.3f}, accuracy {f[f'so_acc_{ctx}']:.3f} "
          f"(mean SO: positive {f[f'so_mean_positive_{ctx}']:+.3f}, negative {f[f'so_mean_negative_{ctx}']:+.3f})")
print("polysemes' SO (document):", {w: round(v, 3) for w, v in f["polyseme_so"].items()},
      f"| median |SO| {f['polyseme_abs_so_median']:.3f} against the lexicon's {f['lexicon_abs_so_median']:.3f}")
print(f"EDGAR: {f['edgar_tokens']} word tokens, {f['edgar_covered']:.1%} with a training vector "
      f"({f['edgar_covered_min_count']:.1%} seen ≥ {ds.MIN_COUNT} times)")

## The lab's numbers, and the full harness

`viz_payload()` holds every number the lab shows. Panel A covers every context × weighting cell (counts, PPMI,
SPPMI₅, and the dev-selected SVD), with each lab word's nearest neighbors and the cell's test scores. Panel B holds
the selected SVD's $(d, p)$ sweep, the L2 spectra of the real and shuffled corpora, the signed eigenvalues, and
Thm 3's two spectra.

In [ ]:
ds.test_viz_payload()
P = ds.viz_payload()
for key in ("L2|counts", "L2|ppmi", "L8|svd", "document|ppmi"):
    cell = P["cells"][key]
    print(key, f"AUC {cell['test_auc']}, 1-NN {cell['test_nn']}:", "loans →", [n for n, _ in cell["neighbors"]["loans"]])
print("SO accuracy by context:", P["so_accuracy"])

In [ ]:
assert ds._run_harness() == 0